In [1]:
import os
import sys

from src.nba_scrapping import *
from src.utils import *
from src.config import *

from datetime import datetime

In [2]:
start_time = datetime.now()

# Retry errors for historical boxscores (safety)

In [3]:
#retry_seasons = ["2012-13"]

#retry_seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2011, 2025)]
retry_seasons = ["2010-11"]


for retry_season in retry_seasons:
    print(f"--- Retry des boxscores échoués pour la saison {retry_season} ---")
    
    season_batch_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, retry_season)
    
    retry_failed_boxscores_for_season(season_batch_dir, max_retries=10)


--- Retry des boxscores échoués pour la saison 2010-11 ---
[INFO] Retrying 0 GAME_IDs for season folder data\raw\boxscores\batches\2010-11
✅ Retry process completed.


# Saisons ciblées


In [8]:
seasons = [f"{y}-{str(y+1)[-2:]}" for y in range(2010, 2025)]
#seasons = ["2010-11"]

seasons


['2010-11',
 '2011-12',
 '2012-13',
 '2013-14',
 '2014-15',
 '2015-16',
 '2016-17',
 '2017-18',
 '2018-19',
 '2019-20',
 '2020-21',
 '2021-22',
 '2022-23',
 '2023-24',
 '2024-25']

# Merge batches and remove duplicate for all endpoints

In [9]:
for season in seasons:
    print(f"--- Merging boxscores endpoints for {season} ---")
    
    season_batch_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, season)
    
    merge_boxscore_batches_for_season(season_batch_dir)


--- Merging boxscores endpoints for 2010-11 ---
[MERGED] traditional saved to data\raw\boxscores\batches\2010-11\merged_batches\merged_traditional.csv (31419 rows)
[MERGED] advanced saved to data\raw\boxscores\batches\2010-11\merged_batches\merged_advanced.csv (31419 rows)
[MERGED] fourfactors saved to data\raw\boxscores\batches\2010-11\merged_batches\merged_fourfactors.csv (31419 rows)
[MERGED] misc saved to data\raw\boxscores\batches\2010-11\merged_batches\merged_misc.csv (31419 rows)
[MERGED] scoring saved to data\raw\boxscores\batches\2010-11\merged_batches\merged_scoring.csv (31419 rows)
[MERGED] usage saved to data\raw\boxscores\batches\2010-11\merged_batches\merged_usage.csv (31419 rows)
--- Merging boxscores endpoints for 2011-12 ---
[MERGED] traditional saved to data\raw\boxscores\batches\2011-12\merged_batches\merged_traditional.csv (27638 rows)
[MERGED] advanced saved to data\raw\boxscores\batches\2011-12\merged_batches\merged_advanced.csv (27638 rows)
[MERGED] fourfactors s

# Merge endpoints csv into one final with all columns


In [10]:
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

for season in seasons:
    print(f"--- Merging all endpoints csv into final for {season} ---")
    
    season_merged_batch_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, season, "merged_batches")
    
    all_boxscores_merged_filepath = merge_all_boxscore_stats(season_merged_batch_dir, output_filename=f"final_merged_all_boxscores_{run_timestamp}.csv")
    
    all_boxscores_merged_df = pd.read_csv(all_boxscores_merged_filepath, dtype={'gameId': str})
    
    #analyze_redundant_columns(all_boxscores_merged_df)
    

--- Merging all endpoints csv into final for 2010-11 ---
Shape before cleaning: (31411, 164)
[CLEAN] Renamed and dropped 55 redundant columns.
[CLEAN] Dropped 9 highly correlated columns.
Shape after cleaning: (31411, 100)
✅ All endpoints merged into: data\raw\boxscores\batches\2010-11\merged_batches\final_merged_all_boxscores_2025-06-06_19-16-58.csv (31411 rows)
--- Merging all endpoints csv into final for 2011-12 ---
Shape before cleaning: (27638, 164)
[CLEAN] Renamed and dropped 55 redundant columns.
[CLEAN] Dropped 9 highly correlated columns.
Shape after cleaning: (27638, 100)
✅ All endpoints merged into: data\raw\boxscores\batches\2011-12\merged_batches\final_merged_all_boxscores_2025-06-06_19-16-58.csv (27638 rows)
--- Merging all endpoints csv into final for 2012-13 ---
Shape before cleaning: (33542, 164)
[CLEAN] Renamed and dropped 55 redundant columns.
[CLEAN] Dropped 9 highly correlated columns.
Shape after cleaning: (33542, 100)
✅ All endpoints merged into: data\raw\boxscor

e:\Documents_\Dev\NBA_Predictor\src\nba_scrapping.py:318: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, dtype={'gameId': str})


Shape before cleaning: (32686, 164)
[CLEAN] Renamed and dropped 55 redundant columns.
[CLEAN] Dropped 9 highly correlated columns.
Shape after cleaning: (32686, 100)
✅ All endpoints merged into: data\raw\boxscores\batches\2018-19\merged_batches\final_merged_all_boxscores_2025-06-06_19-16-58.csv (32686 rows)
--- Merging all endpoints csv into final for 2019-20 ---
Shape before cleaning: (28607, 164)
[CLEAN] Renamed and dropped 55 redundant columns.
[CLEAN] Dropped 9 highly correlated columns.
Shape after cleaning: (28607, 100)
✅ All endpoints merged into: data\raw\boxscores\batches\2019-20\merged_batches\final_merged_all_boxscores_2025-06-06_19-16-58.csv (28607 rows)
--- Merging all endpoints csv into final for 2020-21 ---
Shape before cleaning: (31447, 164)
[CLEAN] Renamed and dropped 55 redundant columns.
[CLEAN] Dropped 9 highly correlated columns.
Shape after cleaning: (31447, 100)
✅ All endpoints merged into: data\raw\boxscores\batches\2020-21\merged_batches\final_merged_all_boxsco

# 🧩 Merge all seasons final boxscores into one


In [11]:
print("\n[MERGE] Concatenating all final_merged_all_boxscores_*.csv into one DataFrame")
final_files = []

for season in seasons:
    season_dir = os.path.join(DATA_BOXSCORES_BATCHES_DIR, season, "merged_batches")
    for file in os.listdir(season_dir):
        if file.startswith("final_merged_all_boxscores_") and file.endswith(".csv"):
            final_files.append(os.path.join(season_dir, file))

if not final_files:
    raise FileNotFoundError("❌ Aucun fichier final_merged_all_boxscores_*.csv trouvé.")

merged_df = pd.concat([pd.read_csv(f, dtype={'gameId': str}) for f in final_files], ignore_index=True)

os.makedirs(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR, exist_ok=True)
output_path = os.path.join(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR, f"all_seasons_boxscores_merged_{run_timestamp}.csv")
merged_df.to_csv(output_path, index=False)
print(f"✅ Sauvegardé dans : {output_path}")


[MERGE] Concatenating all final_merged_all_boxscores_*.csv into one DataFrame
✅ Sauvegardé dans : data\raw_last\batches_merged\all_seasons_boxscores_merged_2025-06-06_19-16-58.csv


# --- Merge games from each season (take most recent per season) ---


In [15]:
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

all_games = []

#use custom format because of the way we store the games by periods or not for scrapping
season_dirs = [dir for dir in os.listdir(DATA_GAMES_DIR) if os.path.isdir(os.path.join(DATA_GAMES_DIR, dir))]

print(season_dirs)

for season_dir in season_dirs:
    season_games_dir = os.path.join(DATA_GAMES_DIR, season_dir)
    if os.path.exists(season_games_dir):
        game_files = [os.path.join(season_games_dir, f) for f in os.listdir(season_games_dir) if f.endswith('.csv')]
        if game_files:
            latest_file = max(game_files, key=os.path.getmtime)
            df_games = pd.read_csv(latest_file, dtype={'GAME_ID': str})
            all_games.append(df_games)

if all_games:
    all_games_df = pd.concat(all_games, ignore_index=True)
    games_output_path = os.path.join(DATA_LAST_GAMES_MERGED_DIR, f"games_merged_all_seasons_{run_timestamp}.csv")
    
    os.makedirs(DATA_LAST_GAMES_MERGED_DIR, exist_ok=True)
    
    all_games_df.to_csv(games_output_path, index=False)
    print(f"✅ Merged games saved to {games_output_path} ({len(all_games_df)} rows)")


['2010-11_2010-11', '2011-12_2011-12', '2012-13_2014-15', '2015-16_2019-20', '2018-19_2018-19', '2019-20_2019-20', '2020-21_2024-25', '2023-24_2023-24', '2024-25_2024-25']
✅ Merged games saved to data\raw_last\games_merged\games_merged_all_seasons_2025-06-06_19-41-46.csv (48504 rows)


In [13]:
#store end time of notebook
end_time = datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-06-06 19:19:03.959121
Total time:  0:15:21.857635


In [14]:
print("\n✅ Scraping historique V3 terminé")



✅ Scraping historique V3 terminé
